# Interview Prep: 03 - Backend Architectures

This notebook focuses on modern Python backend development, focusing on **FastAPI**, **Django**, and **Scalable System Design**. As a senior developer, you are expected to understand not just how to use these frameworks, but the architectural trade-offs involved in performance and scalability.

---

## 1. FastAPI vs. Django: The Great Debate

### **Interview Question**: "When would you choose Django over FastAPI, and vice versa?"

#### **Key Talking Points**:
- **Django**: "Batteries included." Best for standard CRUD applications, admin panels, and rapid development where stability and a robust ecosystem are prioritized.
- **FastAPI**: Lightweight, high performance, asynchronous. Best for microservices, high-concurrency APIs, and projects requiring modern typing (Pydantic).

| Feature | Django | FastAPI |
| :--- | :--- | :--- |
| **ORM** | Built-in (very powerful) | Pluggable (SQLAlchemy, Tortoise, etc.) |
| **Philosophy** | Monolithic / Batteries Included | Modular / Micro |
| **Async** | Added in 3.0+ (partial) | First-class citizen (native) |
| **Validation** | Django Forms / DRF | Pydantic (Type hints) |

## 2. Performance: Sync vs. Async

### **Interview Question**: "Why is `async` often faster for I/O bound tasks but sometimes slower for CPU-bound tasks?"

#### **Scenario**:
Imagine a backend that waits for a Database query (I/O) vs. a backend that calculates a large prime number (CPU).

```python
# Async Example (FastAPI)
@app.get("/data")
async def get_data():
    result = await database.fetch_all("SELECT * FROM items") # Non-blocking
    return result
```

**The Logic**: In `async`, the event loop registers the I/O task and moves on to the next request. For CPU tasks, the event loop is *blocked* because Python is single-threaded (due to the GIL), meaning no other requests can be handled while the calculation is running.

## 3. Dependency Injection in FastAPI

### **Interview Question**: "What are the benefits of using `Depends` in FastAPI?"

**Answer**: Dependency injection allows for better code reusability, easier testing (mocking dependencies), and cleaner separation of concerns (e.g., separating authentication logic from route logic).

#### **Challenge**: Implement a simple Dependency for API Key validation.


In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Header

app = FastAPI()

async def verify_token(x_token: str = Header(...)):
    if x_token != "sceret-token-123":
        raise HTTPException(status_code=400, detail="Invalid Token")
    return x_token

@app.get("/secure-data", dependencies=[Depends(verify_token)])
async def read_secure_data():
    return {"data": "You are authorized!"}

print("Dependency Injection structure ready.")

## 4. The N+1 Problem in Django ORM

### **Interview Question**: "What is the N+1 problem, and how do you solve it in Django?"

**Definition**: It occurs when you fetch a list of objects and then perform a separate query for each object's related field.

**Solution**: 
- `select_related()`: For ForeignKey/OneToOne (SQL JOIN).
- `prefetch_related()`: For ManyToMany/Reverse ForeignKey (separate query + Python-side joining).

---

## 5. Coding Challenge: Custom Middleware

**Task**: Write a FastAPI middleware that logs the time taken for each request.


In [ ]:
import time
from starlette.middleware.base import BaseHTTPMiddleware
from fastapi import Request

class TimingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        start_time = time.time()
        response = await call_next(request)
        process_time = time.time() - start_time
        response.headers["X-Process-Time"] = str(process_time)
        print(f"Request to {request.url.path} took {process_time:.4f}s")
        return response

print("Middleware implementation complete.")